# Загрузка данных TPCH из Trino в MinIO

Этот notebook читает TPCH-таблицы из Trino и выгружает их в MinIO в bucket `raw` в формате Parquet.

## 1. Импорт библиотек и создание SparkSession

Используем Spark для чтения через JDBC из Trino и записи в S3-совместимое хранилище MinIO.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .appName("LoadTPCHToMinIO") \
    .remote("sc://spark-connect:15002") \
    .getOrCreate()

In [3]:
spark.conf.set("fs.s3a.access.key", "minioadmin")
spark.conf.set("fs.s3a.secret.key", "minioadmin")
spark.conf.set("fs.s3a.endpoint", "http://minio:9002")
spark.conf.set("fs.s3a.path.style.access", "true")
spark.conf.set("fs.s3a.connection.ssl.enabled", "false")
spark.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

In [4]:
spark.conf.get("fs.s3a.endpoint")

'http://minio:9002'

In [5]:
spark.conf.get("fs.s3a.access.key")

'minioadmin'

In [6]:
spark.conf.get("fs.s3a.secret.key")

'minioadmin'

## 2. Параметры подключения к Trino и MinIO

In [7]:
trino_host = "trino-coordinator"
trino_port = 8080
trino_catalog = "tpch"
trino_schema = "tiny"
trino_user = "admin"
trino_password = ""

minio_bucket = "raw"
minio_base_path = f"s3a://{minio_bucket}/tpch"

jdbc_url = f"jdbc:trino://{trino_host}:{trino_port}/{trino_catalog}/{trino_schema}"
jdbc_driver = "io.trino.jdbc.TrinoDriver"

print("Trino connection settings:")
print(f"  URL: {jdbc_url}")
print(f"  User: {trino_user}")
print(f"  Schema: {trino_schema}")
print("MinIO settings:")
print(f"  Bucket: {minio_bucket}")
print(f"  Base path: {minio_base_path}")

Trino connection settings:
  URL: jdbc:trino://trino-coordinator:8080/tpch/tiny
  User: admin
  Schema: tiny
MinIO settings:
  Bucket: raw
  Base path: s3a://raw/tpch


## 3. Список TPCH-таблиц

In [8]:
tpch_tables = ["nation", "region", "customer", "orders", "lineitem", "part", "partsupp", "supplier"]
print(tpch_tables)

['nation', 'region', 'customer', 'orders', 'lineitem', 'part', 'partsupp', 'supplier']


## 4. Создание bucket `raw`, если он еще не существует

In [9]:
import boto3
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9002",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
)

try:
    s3.head_bucket(Bucket=minio_bucket)
    print(f"Bucket `{minio_bucket}` already exists")
except ClientError:
    s3.create_bucket(Bucket=minio_bucket)
    print(f"Bucket `{minio_bucket}` created")

Bucket `raw` already exists


## 5. Чтение таблиц из Trino и запись в MinIO

In [10]:
def read_trino_table(table_name: str):
    return spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("user", trino_user) \
        .option("password", trino_password) \
        .option("driver", jdbc_driver) \
        .load()

for table_name in tpch_tables:
    print(f"Processing {table_name}...")
    df = read_trino_table(table_name)
    row_count = df.count()
    target_path = f"{minio_base_path}/{table_name}"
    print(target_path)

    df.write \
        .mode("overwrite") \
        .parquet(target_path)

    print(f"  rows: {row_count}")
    print(f"  written to: {target_path}")

Processing nation...
s3a://raw/tpch/nation
  rows: 25
  written to: s3a://raw/tpch/nation
Processing region...
s3a://raw/tpch/region
  rows: 5
  written to: s3a://raw/tpch/region
Processing customer...
s3a://raw/tpch/customer
  rows: 1500
  written to: s3a://raw/tpch/customer
Processing orders...
s3a://raw/tpch/orders
  rows: 15000
  written to: s3a://raw/tpch/orders
Processing lineitem...
s3a://raw/tpch/lineitem
  rows: 60175
  written to: s3a://raw/tpch/lineitem
Processing part...
s3a://raw/tpch/part
  rows: 2000
  written to: s3a://raw/tpch/part
Processing partsupp...
s3a://raw/tpch/partsupp
  rows: 8000
  written to: s3a://raw/tpch/partsupp
Processing supplier...
s3a://raw/tpch/supplier
  rows: 100
  written to: s3a://raw/tpch/supplier


## 6. Проверка результата

In [9]:
sample_table = tpch_tables[0]
check_df = spark.read.parquet(f"{minio_base_path}/{sample_table}")
print(f"Sample table: {sample_table}")
print(f"Rows in MinIO: {check_df.count()}")
check_df.show(5, truncate=False)

Sample table: nation
Rows in MinIO: 25
+---------+---------+---------+-----------------------------------------------------------------------------------------------------------+
|nationkey|name     |regionkey|comment                                                                                                    |
+---------+---------+---------+-----------------------------------------------------------------------------------------------------------+
|0        |ALGERIA  |0        | haggle. carefully final deposits detect slyly agai                                                        |
|1        |ARGENTINA|1        |al foxes promise slyly according to the regular accounts. bold requests alon                               |
|2        |BRAZIL   |1        |y alongside of the pending deposits. carefully special packages are about the ironic forges. slyly special |
|3        |CANADA   |1        |eas hang ironic, silent packages. slyly regular packages are furiously over the tithes. fl